# Task 3 (D3 Phase A) — Answerer: citations, refusal & the strategy comparison

**Owner: WAFIQ.** Deliverable artifact for the zero-shot answerer. The *logic* lives in
`src/csai415/answer.py` (importable + tested); this notebook only **runs** it and renders the
evidence:

1. one grounded answer end-to-end (smoke),
2. the citation-strategy comparison — **numbered vs hybrid vs posthoc** × **Qwen-3B vs Groq-70B ceiling**,
   scored on a faithfulness proxy × p95 latency.

Backend + strategy are env-driven (`CSAI415_ANSWERER`, `CSAI415_CITE_MODE`) — no code changes to swap.
Develop against **Groq** (free, fast, no GPU); produce the graded **Qwen-3B** numbers on the Colab GPU.

## 0 · Environment (Colab clones the repo; local is a no-op)

In [ ]:
import sys, subprocess, os
from pathlib import Path

REPO_URL = "https://github.com/waf-iq/special-topics.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path("special-topics").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("special-topics")
    subprocess.run(["git", "pull", "--ff-only"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    # `pip install -e .` registers csai415 via a .pth file, but .pth files are only read at
    # interpreter startup — a live kernel won't see it. Put src on the path directly so the
    # import works this session without a runtime restart.
    sys.path.insert(0, str(Path.cwd() / "src"))
else:
    # local: assume the notebook runs from repo root or notebooks/ — make src importable
    root = Path.cwd()
    if (root / "notebooks").exists() is False and (root.parent / "src").exists():
        os.chdir(root.parent)
    sys.path.insert(0, str(Path.cwd() / "src"))

print("cwd:", Path.cwd())

In [ ]:
# Backend config. For dev: Groq (free, instant, no GPU). For graded Qwen numbers on the
# Colab GPU: start Ollama + `ollama pull qwen2.5:3b-instruct`, then set CSAI415_ANSWERER to it.
from getpass import getpass

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("GROQ_API_KEY (free, console.groq.com): ")

os.environ["CSAI415_ANSWERER"] = "groq:llama-3.3-70b-versatile"  # dev backend
os.environ["CSAI415_CITE_MODE"] = "numbered"

from csai415 import answer as ans
print("backend:", ans.current_backend())

## 1 · Smoke: one grounded answer

`generate_answer(query, citations, contexts)` with `contexts[i]` aligned to `citations[i]`.
Here we pass a tiny hand-made context list; swap in the real executor output once Task 1/2/7
land (`GraphRAGExecutor.answer(...)` returns aligned `.citations` / `.contexts`).

In [ ]:
from csai415.graphrag import Citation

# fixture (query, contexts) — replace with real retrieved chunks via the executor
query = "What problem does the attention mechanism solve in sequence models?"
contexts = [
    "Attention lets a model weigh all input positions when producing each output token, "
    "removing the fixed-length bottleneck of RNN encoder-decoders on long sequences.",
    "BERT is a bidirectional encoder pre-trained with masked language modeling.",
    "Stochastic gradient descent updates parameters using mini-batches.",
]
citations = [Citation(f"arxiv:p{i}:0", f"p{i}", f"Paper {i}", "1-2", 1.0) for i in range(len(contexts))]

text, used = ans.generate_answer(query, citations, contexts)
print("ANSWER:\n", text)
print("\nCITED:", used, "->", [citations[i - 1].chunk_id for i in used])

## 1b · Serve Qwen-3B locally via Ollama  *(GPU runtime only)*

Run this **only on a GPU runtime** (Runtime → Change runtime type → GPU). It installs Ollama,
serves it in the background, and pulls `qwen2.5:3b-instruct` (the zero-shot answerer / D4 QLoRA
base). On CPU it technically works but is ~10–40s per answer — far over the 2s p95 target — so
skip it on a CPU session and just read the Groq rows. Once this finishes, re-run the comparison
cell and the `qwen-3b` rows will populate.

In [ ]:
import subprocess, time, os, requests

# 0. GPU check (warn, don't block)
try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
except Exception:
    print("(torch not importable — can't check GPU)")

# 1. Install Ollama (no-op if already on PATH)
if subprocess.run(["which", "ollama"], capture_output=True).returncode != 0:
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

# 2. Start the server in the background (harmless if one is already running)
os.environ.setdefault("OLLAMA_HOST", "127.0.0.1:11434")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 3. Wait until it answers
for _ in range(60):
    try:
        if requests.get("http://127.0.0.1:11434/api/tags", timeout=2).ok:
            break
    except Exception:
        pass
    time.sleep(1)

# 4. Pull the zero-shot answerer (the D4 QLoRA base). ~1.9 GB, one-time per session.
subprocess.run(["ollama", "pull", "qwen2.5:3b-instruct"], check=True)
print("Ollama ready; qwen2.5:3b-instruct pulled. Re-run the comparison cell.")

## 2 · Strategy × backend comparison (the deep-dive table)

Sweep `CSAI415_CITE_MODE` × `CSAI415_ANSWERER`. `faith_proxy` = lexical overlap of the answer
with the cited contexts (a cheap stand-in until Ahmad's RAGAS harness lands — replace the
`faith_proxy` column with the real RAGAS faithfulness row at integration). Latency is wall-clock
per call. Missing backends (e.g. no local Ollama) are skipped, not fatal.

In [ ]:
from csai415.graphrag import Citation

# Hand-authored DEV examples for the *strategy* comparison (numbered vs hybrid vs posthoc).
# Each = one question + candidate contexts (1 relevant + distractors drawn from the topic pool,
# so a weaker model has something to miscite — that's where the hybrid verifier earns its keep).
#
# NOTE: this is NOT the graded gold set. The real eval set is Ahmad's data/gold/qa_answers.jsonl
# (Task 4), and the final numbers run on real executor output over the arXiv corpus. This set
# only exists to measure how the three citation strategies *behave* against each other.

def make(question, contexts):
    cits = [Citation(f"arxiv:p{i}:0", f"p{i}", f"Source {i + 1}", "1-2", 1.0) for i in range(len(contexts))]
    return (question, cits, contexts)

QUESTIONS = [
    make("What problem does the attention mechanism solve in sequence-to-sequence models?", [
        "Attention lets the decoder weigh all encoder positions when producing each output token, "
        "removing the fixed-length context-vector bottleneck that limited RNN encoder-decoders on long sequences.",
        "BERT is a bidirectional Transformer encoder pre-trained with a masked language modeling objective.",
        "Stochastic gradient descent updates model parameters using gradients estimated from mini-batches.",
    ]),
    make("How is BERT pre-trained?", [
        "BERT is pre-trained with two self-supervised objectives: masked language modeling, which predicts "
        "randomly masked tokens, and next-sentence prediction.",
        "Beam search keeps the k highest-scoring partial sequences at each decoding step instead of committing to one token.",
        "Dropout randomly zeroes a fraction of activations during training to reduce overfitting.",
    ]),
    make("What is dropout and why is it used?", [
        "Dropout randomly sets a fraction of unit activations to zero during each training step, reducing "
        "co-adaptation of neurons and acting as a regularizer against overfitting.",
        "BLEU compares machine-translation output to reference translations using modified n-gram precision with a brevity penalty.",
        "Word2vec learns dense word vectors such that words appearing in similar contexts end up close in the embedding space.",
    ]),
    make("How does the BLEU metric evaluate machine translation?", [
        "BLEU scores a candidate translation by modified n-gram precision against one or more references, "
        "multiplied by a brevity penalty that discourages overly short outputs.",
        "Layer normalization normalizes the summed inputs to each neuron across the feature dimension, stabilizing deep-network training.",
        "Attention weights are computed as a softmax over query-key dot products.",
    ]),
    make("What does word2vec learn about words?", [
        "Word2vec produces dense vector representations in which words occurring in similar contexts are placed "
        "near each other, capturing distributional semantic similarity.",
        "Next-sentence prediction trains a model to decide whether one sentence follows another in the original text.",
        "Beam search is a heuristic decoding strategy that explores several high-probability sequences in parallel.",
    ]),
    make("Why is beam search used during decoding?", [
        "Beam search keeps the k highest-scoring partial sequences at each step, trading extra computation for "
        "higher output quality than greedy decoding, which commits to a single token at a time.",
        "Dropout is disabled at inference time; the full network is used with appropriately scaled weights.",
        "BLEU applies a brevity penalty to penalize translations shorter than the reference.",
    ]),
    make("What is the purpose of layer normalization in Transformers?", [
        "Layer normalization rescales the activations within each layer across the feature dimension, which "
        "stabilizes and speeds up training of deep Transformer networks.",
        "Masked language modeling predicts tokens that have been randomly hidden from the input.",
        "Word2vec embeddings can be trained with either the skip-gram or continuous-bag-of-words objective.",
    ]),
]

print(f"{len(QUESTIONS)} dev questions loaded")

In [ ]:
import re, time
import numpy as np, pandas as pd

def faith_proxy(answer: str, ctxs) -> float:
    aw = set(re.findall(r"[a-z]{3,}", (answer or "").lower()))
    bw = set(re.findall(r"[a-z]{3,}", " ".join(ctxs).lower()))
    return len(aw & bw) / len(aw) if aw else 0.0

# (label, CSAI415_ANSWERER). Add the tuned model once D4 lands: ("qwen-tuned", "qwen2.5-3b-csai415").
BACKENDS = [
    ("groq-70b", "groq:llama-3.3-70b-versatile"),
    ("qwen-3b", "qwen2.5:3b-instruct"),
]
STRATEGIES = ["numbered", "hybrid", "posthoc"]  # QUESTIONS comes from the cell above

rows = []
for blabel, bspec in BACKENDS:
    os.environ["CSAI415_ANSWERER"] = bspec
    for strat in STRATEGIES:
        os.environ["CSAI415_CITE_MODE"] = strat
        lats, faiths, ncite = [], [], []
        try:
            for q, cits, ctxs in QUESTIONS:
                t0 = time.perf_counter()
                a, u = ans.generate_answer(q, cits, ctxs)
                lats.append((time.perf_counter() - t0) * 1000)
                faiths.append(faith_proxy(a, [ctxs[i - 1] for i in u] or ctxs))
                ncite.append(len(u))
        except Exception as e:
            rows.append({"backend": blabel, "strategy": strat, "status": f"skip: {type(e).__name__}"})
            continue
        rows.append({
            "backend": blabel, "strategy": strat, "status": "ok",
            "faith_proxy": round(float(np.mean(faiths)), 3),
            "avg_citations": round(float(np.mean(ncite)), 2),
            "p95_latency_ms": round(float(np.percentile(lats, 95)), 1),
        })

pd.DataFrame(rows)

## 3 · Verdict

_Fill in after running on the Colab GPU with the real Qwen-3B:_

- Winning strategy and why (faithfulness vs p95 trade-off).
- Qwen-3B vs Groq-70B ceiling gap.
- Whether the 2s p95 target holds at the chosen `max_tokens` / context truncation.
- Note the `faith_proxy` caveat: it scores the answer against the *cited* contexts, so a miscited
  distractor can still look 'faithful' — read it alongside `avg_citations`, and replace it with
  Ahmad's RAGAS faithfulness at integration.
- AI log: approaches compared + share link (per the deliverable checklist).